In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

from constants import CA_NAD83_Albers, CULVER_CITY_FEED_KEY, SERVICE_DATE, SHAPE_KEY_TO_SHAPE_ID_MAP
from _data_loaders import (
    get_culver_city_vehicle_positions,
    get_selected_shapes,
    get_traffic_signals,
    list_available_service_dates,
)

import shapely
from constants import MAX_SNAP_DISTANCE_M
from match_shapes_vp import project_points_on_shape

In [ ]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]
SEGMENT_START_SIGNAL = 29
SEGMENT_END_SIGNAL = SEGMENT_START_SIGNAL + 1
SEGMENT_BUFFER_LENGTH = 50

In [ ]:
service_dates = list_available_service_dates()
vehicle_positions = pd.concat(
    [
        get_culver_city_vehicle_positions([SHAPE_KEY], service_date).assign(service_date=service_date)
        for service_date in service_dates
    ],
    ignore_index=True,
)
print(f"{len(vehicle_positions):,} positions over {len(service_dates)} dates")

shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])
signals = get_traffic_signals()
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

In [ ]:
shape = shapes.iloc[0] # TODO: everything should assume a single shape here

# get start signal
start_signal_geometry = signals.geometry.loc[SEGMENT_START_SIGNAL]
end_signal_geometry = signals.geometry.loc[SEGMENT_END_SIGNAL]

# get signal geometries
signal_distances = project_points_on_shape(signals.geometry, shapes.geometry, MAX_SNAP_DISTANCE_M)
start_signal_distance = signal_distances.loc[SEGMENT_START_SIGNAL]
end_signal_distance = signal_distances.loc[SEGMENT_START_SIGNAL + 1]
start_segment_distance = start_signal_distance - SEGMENT_BUFFER_LENGTH
end_segment_distance = end_signal_distance + SEGMENT_BUFFER_LENGTH

# get segment geometry
segment_geometry = shapely.ops.substring(shape.geometry, start_segment_distance, end_segment_distance)
gdf_segment = gpd.GeoSeries([segment_geometry], crs=3310)

In [ ]:
# get pings along the segment - first, do a filter using geometry because projecting is more expensive
buffered_segment = gpd.GeoSeries([segment_geometry], crs=3310).buffer(500)

In [ ]:
from smooth_trajectory import smooth_distances_per_trip


vehicle_positions_near_segment = vehicle_positions.loc[vehicle_positions.within(buffered_segment.iloc[0])] 
# we want 0 to be the start signal, so these go negative if they are before it (in the buffer)
vehicle_position_distances = project_points_on_shape(vehicle_positions_near_segment, gdf_segment, MAX_SNAP_DISTANCE_M) - SEGMENT_BUFFER_LENGTH

# reference positions in the same coordinate system (0 = start signal)
segment_length = end_signal_distance - start_signal_distance
stop_distances = project_points_on_shape(stops, gdf_segment, MAX_SNAP_DISTANCE_M) - SEGMENT_BUFFER_LENGTH
stops_in_segment = stop_distances[(stop_distances >= -SEGMENT_BUFFER_LENGTH) & (stop_distances <= segment_length + SEGMENT_BUFFER_LENGTH)]
print(f"start signal at 0 m, end signal at {segment_length:.0f} m")
print(f"{len(stops_in_segment)} stop(s) in segment at: {stops_in_segment.round(1).to_list()} m")

ax = vehicle_position_distances.hist(bins=20)
ax.axvline(0, color="green", linestyle="--", label="start signal (segment start)")
ax.axvline(segment_length, color="red", linestyle="--", label="end signal (segment end)")
for i, stop_distance in enumerate(stops_in_segment):
    ax.axvline(stop_distance, color="orange", linestyle=":", label="stop" if i == 0 else None)
ax.set_xlabel("distance along segment (m), 0 = start signal")
ax.legend()

In [ ]:
vp_smoothed = smooth_distances_per_trip(vehicle_positions_near_segment, vehicle_positions_near_segment["distance_along_shape"], freq_seconds=1.0)


In [ ]:
vehicle_positions_near_segment